# Reproduce emotion2vec (ACL Findings 2024) — RAVDESS linear probe

Frozen **emotion2vec_base** (+ **WavLM-Large** comparator) + a **SUPERB linear probe**, **random 10-fold CV (80/10/10), all 1440 speech clips, 8 classes**, reporting **WA/UA/WF1** against the paper's Table 3 (emotion2vec RAVDESS = WA 82.43 / UA 82.86 / WF1 82.39).
Protocol + sources: `docs/tasks/emotion2vec-reproduction.md`.

## 0. Install pinned audio stack  (run once)

In [ ]:
# Cell 0 — pinned audio stack (P100 = sm_60; Kaggle default torch drops it -> crash).
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
# torchvision pinned to match torch 2.5.1 — else transformers' lazy torchvision import
# fails with "operator torchvision::nms does not exist" and WavLM won't load.
get_ipython().system('pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121')
get_ipython().system('pip install -q transformers==4.48.2 datasets==3.2.0 librosa soundfile scikit-learn scipy')
# emotion2vec route (primary/repro backbone). Heavy dep tree; usage is guarded so a
# failed install/import only skips the emotion2vec arm — WavLM still produces a result.
get_ipython().system('pip install -q funasr modelscope || echo "funasr install failed -> emotion2vec arm will be skipped"')
print("install cell done")


## 1. Imports & config

In [ ]:
# Cell 1 — imports & config.
# REPRODUCTION of emotion2vec (Ma et al., ACL Findings 2024), Table 3 RAVDESS row.
#   Frozen encoder + SUPERB linear probe -> WA/UA/WF1, compared to the paper's
#   emotion2vec RAVDESS = WA 82.43 / UA 82.86 / WF1 82.39.
# Protocol (researched, see docs/tasks/emotion2vec-reproduction.md):
#   - all 1440 RAVDESS speech clips, all 8 classes
#   - random 10-fold CV, each fold a fresh stratified 80/10/10 train/val/test split
#   - SUPERB head: Linear(d,256) -> ReLU -> Linear(256,8), frozen single-layer feats, CE
#   - report mean +/- std of per-fold test WA/UA/WF1
import os, json, random, warnings, tempfile
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ART = "/kaggle/working"
SR = 16000
MAX_SEC = 5.0                          # RAVDESS clips ~3-5s; 5s covers virtually all
MAX_SAMPLES = int(SR * MAX_SEC)
FRAME_HOP = 320                        # both encoders downsample 16kHz->50Hz (320x)
T_MAX = MAX_SAMPLES // FRAME_HOP        # 250 frames; v2 keeps frame-level feats (T x d)

N_FOLDS = 10                           # paper: random 10-fold CV
SPLIT = (0.80, 0.10, 0.10)            # paper: 80/10/10 train/val/test per fold
BASE_SEED = 42                         # emotion2vec downstream config seed
PROBE_EPOCHS, PROBE_LR, HEAD_DIM, WD = 100, 1e-3, 256, 1e-4

# RAVDESS 8 emotions, canonical id order (filename code 01..08 -> 0..7).
EMOTIONS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
EMO2ID = {e: i for i, e in enumerate(EMOTIONS)}
N_EMO = len(EMOTIONS)

# Paper Table 3 reference (emotion2vec, RAVDESS, frozen linear probe).
PAPER = {"emotion2vec": {"WA": 82.43, "UA": 82.86, "WF1": 82.39},
         # paper Table 3 comparators (different WavLM variant: base, not large):
         "wavlm-base(paper)": {"WA": 37.01, "UA": None, "WF1": None},
         "data2vec2.0(paper)": {"WA": 81.04, "UA": None, "WF1": None}}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"emotions={EMOTIONS}\nfolds={N_FOLDS} split={SPLIT} base_seed={BASE_SEED} "
      f"probe(ep={PROBE_EPOCHS},lr={PROBE_LR},hid={HEAD_DIM}) | v2 frame-level T_MAX={T_MAX}")


## 2. RAVDESS data  (all 1440 clips, resample 48k->16k, no fixed split)

In [ ]:
# Cell 2 — RAVDESS load (all 1440 speech clips), resample 48k->16k. No fixed split:
# folds are drawn randomly in the probe cell (paper protocol).
import torchaudio
from datasets import load_dataset
import soundfile as sf

ds = load_dataset("narad/ravdess", split="train", trust_remote_code=True)
print("RAVDESS:", ds)
INT2STR = ds.features["labels"].int2str

def fix_len(wav):
    # pad/truncate to 5s; also return the true (pre-pad) valid frame count for masking.
    n = min(len(wav), MAX_SAMPLES)
    flen = int(np.clip(round(n / FRAME_HOP), 1, T_MAX))
    if len(wav) >= MAX_SAMPLES:
        return wav[:MAX_SAMPLES], flen
    return np.pad(wav, (0, MAX_SAMPLES - len(wav))), flen

def to_records(ds):
    recs = []
    for ex in ds:
        a = ex["audio"]
        wav = np.asarray(a["array"], dtype=np.float32)
        src_sr = a["sampling_rate"]
        if src_sr != SR:
            wav = torchaudio.functional.resample(torch.from_numpy(wav), src_sr, SR).numpy()
        wav, flen = fix_len(wav.astype(np.float32))
        emo = INT2STR(ex["labels"])
        recs.append({"wav": wav, "y": EMO2ID[emo], "flen": flen})
    return recs

recs = to_records(ds)
y_all = np.array([r["y"] for r in recs])
print(f"clips={len(recs)} | per-class counts={np.bincount(y_all, minlength=N_EMO).tolist()}")
assert len(recs) == 1440, f"expected 1440 RAVDESS speech clips, got {len(recs)}"

# stash one clip as the local FastAPI/test sample (16 kHz mono wav).
sample = recs[0]
sf.write(f"{ART}/sample_val.wav", sample["wav"], SR)
print(f"sample wav -> {ART}/sample_val.wav (emo={EMOTIONS[sample['y']]})")


## 3. Frozen feature extraction  (each encoder runs once over all clips)

In [ ]:
# Cell 3 — frozen FRAME-LEVEL feature extraction (v2, faithful SUPERB recipe).
# Each encoder runs ONCE over all 1440 clips, returning (N, T_MAX, dim) frame features
# (padded/truncated to T_MAX); the probe pools AFTER the first Linear+ReLU using the
# true frame lengths (c2 `flen`) as a mask — matching emotion2vec/EmoBox SuperbBaseModel.
# emotion2vec = funasr granularity="frame" (T,768); WavLM-Large = last_hidden_state (T,1024).
from transformers import AutoFeatureExtractor, WavLMModel
import soundfile as sf

def pad_T(arr):
    # arr (T, dim) -> (T_MAX, dim) padded with zeros or truncated.
    if len(arr) >= T_MAX:
        return arr[:T_MAX]
    return np.pad(arr, ((0, T_MAX - len(arr)), (0, 0)))

def wavlm_extractor():
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
    model = WavLMModel.from_pretrained("microsoft/wavlm-large").to(DEVICE).eval()
    @torch.no_grad()
    def extract(wavs, bs=16):
        out = []
        for i in range(0, len(wavs), bs):
            batch = [w for w in wavs[i:i + bs]]
            enc = fe(batch, sampling_rate=SR, return_tensors="pt", padding=True)
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            with torch.cuda.amp.autocast(enabled=DEVICE.type == "cuda"):
                h = model(**enc).last_hidden_state            # (B, T, 1024)
            h = h.float().cpu().numpy()
            for j in range(h.shape[0]):
                out.append(pad_T(h[j]).astype(np.float16))    # keep time dim
        return np.stack(out)                                  # (N, T_MAX, 1024)
    return extract, 1024

def emotion2vec_extractor():
    from funasr import AutoModel as FunASR
    try:
        model = FunASR(model="iic/emotion2vec_base", hub="hf", disable_update=True)
    except Exception:
        from huggingface_hub import snapshot_download
        local = snapshot_download("emotion2vec/emotion2vec_base")
        model = FunASR(model=local, disable_update=True)
    def extract(wavs, bs=None):
        feats = []
        for w in wavs:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                sf.write(tf.name, w, SR); path = tf.name
            r = model.generate(path, granularity="frame", extract_embedding=True,
                               output_dir=None)
            f = np.asarray(r[0]["feats"], dtype=np.float32)    # (T, 768)
            os.remove(path)
            feats.append(pad_T(f).astype(np.float16))
        return np.stack(feats)                                 # (N, T_MAX, 768)
    return extract, 768

all_wavs = [r["wav"] for r in recs]
flen_all = np.array([r["flen"] for r in recs])
BACKBONES = {}   # name -> {"X": (N,T_MAX,dim) f16, "dim": int}
for name, builder in [("emotion2vec", emotion2vec_extractor), ("wavlm-large", wavlm_extractor)]:
    try:
        print(f"\n[{name}] building extractor...")
        extract, dim = builder()
        X = extract(all_wavs)
        BACKBONES[name] = {"X": X, "dim": dim}
        print(f"[{name}] frame features: {X.shape} ({X.dtype}) | flen mean={flen_all.mean():.0f}")
        del extract; torch.cuda.empty_cache()
    except Exception as e:
        print(f"[{name}] SKIPPED -> {type(e).__name__}: {e}")

assert BACKBONES, "no backbone produced features"
print("\nbackbones with features:", list(BACKBONES))


## 4. SUPERB linear probe  (random 10-fold CV, WA/UA/WF1)

In [ ]:
# Cell 4 — SUPERB linear probe (v2, frame-level) + random 10-fold CV.
#   head = Linear(d,256) -> ReLU -> MASKED-mean-pool over valid frames -> Linear(256,8).
#   This is the faithful EmoBox/emotion2vec SuperbBaseModel (pool AFTER the first linear).
#   per fold: stratified 80/10/10 split (seeded), train on 80, best epoch by val acc,
#   report WA/UA/WF1 on 10% test. Aggregate mean +/- std over 10 folds.
class SuperbHead(nn.Module):
    def __init__(self, dim, n, hid=HEAD_DIM):
        super().__init__()
        self.pre = nn.Linear(dim, hid)
        self.post = nn.Linear(hid, n)
    def forward(self, x, mask):                              # x (B,T,dim), mask (B,T)
        h = F.relu(self.pre(x))                              # (B,T,hid)
        m = mask.unsqueeze(-1).float()
        h = (h * m).sum(1) / m.sum(1).clamp(min=1.0)         # masked mean over time
        return self.post(h)

def split_80_10_10(y, seed):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=SPLIT[1] + SPLIT[2], random_state=seed, stratify=y)
    rel = SPLIT[2] / (SPLIT[1] + SPLIT[2])
    va, te = train_test_split(tmp, test_size=rel, random_state=seed, stratify=y[tmp])
    return tr, va, te

def metrics(y_true, y_pred):
    return {"WA": 100 * accuracy_score(y_true, y_pred),
            "UA": 100 * recall_score(y_true, y_pred, average="macro", zero_division=0),
            "WF1": 100 * f1_score(y_true, y_pred, average="weighted", zero_division=0)}

def train_one(Xg, mask_g, y, dim, seed):
    tr, va, te = split_80_10_10(y, seed)
    set_seed(seed)
    yt = torch.tensor(y, dtype=torch.long, device=DEVICE)
    head = SuperbHead(dim, N_EMO).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=PROBE_LR, weight_decay=WD)
    tr_t = torch.tensor(tr, device=DEVICE)
    best_val, best_state = -1.0, None
    for _ in range(PROBE_EPOCHS):
        head.train(); opt.zero_grad()
        loss = F.cross_entropy(head(Xg[tr_t], mask_g[tr_t]), yt[tr_t])
        loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            va_pred = head(Xg[va], mask_g[va]).argmax(-1).cpu().numpy()
        va_acc = accuracy_score(y[va], va_pred)
        if va_acc > best_val:
            best_val = va_acc
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
    head.load_state_dict(best_state); head.eval()
    with torch.no_grad():
        te_pred = head(Xg[te], mask_g[te]).argmax(-1).cpu().numpy()
    return metrics(y[te], te_pred), head

# frame mask (N, T_MAX): True for valid frames.
mask_all = (torch.arange(T_MAX)[None, :] < torch.tensor(flen_all)[:, None]).to(DEVICE)

results = {}     # name -> list of per-fold metric dicts
artifacts = {}   # name -> head from fold 0
for name, b in BACKBONES.items():
    Xg = torch.tensor(b["X"], dtype=torch.float, device=DEVICE)   # (N,T,dim) f16->f32 on GPU
    results[name] = []
    for fold in range(N_FOLDS):
        m, head = train_one(Xg, mask_all, y_all, b["dim"], BASE_SEED + fold)
        results[name].append(m)
        if fold == 0:
            artifacts[name] = head
        print(f"[{name} fold={fold}] WA={m['WA']:.2f} UA={m['UA']:.2f} WF1={m['WF1']:.2f}")
    del Xg; torch.cuda.empty_cache()
    arr = {k: np.array([f[k] for f in results[name]]) for k in ("WA", "UA", "WF1")}
    print(f"[{name}] MEAN  WA={arr['WA'].mean():.2f}±{arr['WA'].std():.2f} "
          f"UA={arr['UA'].mean():.2f}±{arr['UA'].std():.2f} "
          f"WF1={arr['WF1'].mean():.2f}±{arr['WF1'].std():.2f}")


## 5. Results vs paper + artifacts

In [ ]:
# Cell 5 — aggregate (mean +/- std over folds), compare to paper Table 3, save artifacts.
import pandas as pd
HF_ID = {"wavlm-large": "microsoft/wavlm-large", "emotion2vec": "iic/emotion2vec_base"}

rows, agg = [], {}
for name, folds in results.items():
    agg[name] = {}
    for k in ("WA", "UA", "WF1"):
        vals = np.array([f[k] for f in folds])
        agg[name][k] = (float(vals.mean()), float(vals.std()))
        rows.append({"backbone": name, "metric": k,
                     "mean": round(float(vals.mean()), 2), "std": round(float(vals.std()), 2),
                     "paper": PAPER.get(name, {}).get(k)})
df = pd.DataFrame(rows)
df.to_csv(f"{ART}/results_emotion2vec_repro.csv", index=False)
print(df.to_string(index=False))

# headline comparison vs the paper's emotion2vec RAVDESS row.
print("\n=== Reproduction vs paper (emotion2vec, RAVDESS) ===")
for k in ("WA", "UA", "WF1"):
    if "emotion2vec" in agg:
        ours = agg["emotion2vec"][k][0]; paper = PAPER["emotion2vec"][k]
        print(f"  {k}: ours={ours:.2f}  paper={paper:.2f}  delta={ours - paper:+.2f}")
if "emotion2vec" in agg and "wavlm-large" in agg:
    d = agg["emotion2vec"]["WA"][0] - agg["wavlm-large"]["WA"][0]
    print(f"\nThesis comparator (our run): emotion2vec - wavlm-large WA delta = {d:+.2f}")

# save a serving/test bundle per backbone (fold-0 head).
for name, head in artifacts.items():
    d = f"{ART}/artifact_{name}"
    os.makedirs(d, exist_ok=True)
    torch.save(head.state_dict(), f"{d}/emotion_head.pt")
    cfg = {"backbone_hf_id": HF_ID[name], "backbone": name, "embed_dim": BACKBONES[name]["dim"],
           "emotions": EMOTIONS, "sample_rate": SR, "max_samples": MAX_SAMPLES,
           "frame_hop": FRAME_HOP, "t_max": T_MAX,
           "pooling": "masked-mean", "head": "superb-frame", "head_dim": HEAD_DIM}
    with open(f"{d}/config.json", "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"  saved bundle -> {d}")

with open(f"{ART}/results_emotion2vec_repro.json", "w") as f:
    json.dump({"agg": agg, "per_fold": results, "paper": PAPER,
               "n_folds": N_FOLDS, "split": SPLIT, "base_seed": BASE_SEED}, f, indent=2)
print("\nartifacts in /kaggle/working: results_emotion2vec_repro.{csv,json}, sample_val.wav, artifact_*/")


## 6. Demo inference on held-out clips  (tests emotion2vec too)

In [ ]:
# Cell 6 — demo inference on held-out fold-0 test clips, for EVERY backbone that ran
# (incl. emotion2vec, which can't run locally without funasr). Reuses the already-extracted
# frame features + the fold-0 trained head — no re-extraction. Prints a table and saves
# demo_predictions.json + demo.html so the result is viewable from the Kaggle output.
DEMO_PER_CLASS = 2
tr0, va0, te0 = split_80_10_10(y_all, BASE_SEED + 0)        # fold-0 test = unseen by the head
sel = []
for c in range(N_EMO):
    sel += [i for i in te0 if y_all[i] == c][:DEMO_PER_CLASS]
sel = sorted(sel)
print(f"=== Demo inference on {len(sel)} held-out fold-0 test clips ===")

rows = []
for name, head in artifacts.items():
    Xg = torch.tensor(BACKBONES[name]["X"][sel], dtype=torch.float, device=DEVICE)
    mg = mask_all[sel]
    head.eval()
    with torch.no_grad():
        probs = torch.softmax(head(Xg, mg), dim=-1).cpu().numpy()
    print(f"\n[{name}]")
    for k, idx in enumerate(sel):
        p = probs[k]; top = int(p.argmax()); ok = top == int(y_all[idx])
        rows.append({"backbone": name, "idx": int(idx), "truth": EMOTIONS[y_all[idx]],
                     "pred": EMOTIONS[top], "p": round(float(p[top]), 3), "correct": ok})
        print(f"  {'OK ' if ok else 'XX '} idx={idx:4d} truth={EMOTIONS[y_all[idx]]:9s} "
              f"pred={EMOTIONS[top]:9s} p={p[top]:.3f}")
    acc = np.mean([r["correct"] for r in rows if r["backbone"] == name])
    print(f"  demo acc ({len(sel)} clips) = {acc:.2f}")

with open(f"{ART}/demo_predictions.json", "w") as f:
    json.dump(rows, f, indent=2)

# minimal static HTML view (download from the kernel output and open locally).
def _html(rows):
    head = ("<style>body{font:14px system-ui;background:#0f1419;color:#e6edf3;padding:20px}"
            "table{border-collapse:collapse}td,th{border:1px solid #2a3340;padding:6px 10px}"
            ".ok{color:#3fb950}.xx{color:#d29922}</style>")
    body = "<h2>emotion2vec reproduction — demo predictions (Kaggle)</h2>"
    for name in sorted({r["backbone"] for r in rows}):
        rs = [r for r in rows if r["backbone"] == name]
        acc = np.mean([r["correct"] for r in rs])
        body += f"<h3>{name} — demo acc {acc:.2f}</h3><table><tr><th>idx</th><th>truth</th><th>pred</th><th>p</th></tr>"
        for r in rs:
            cls = "ok" if r["correct"] else "xx"
            body += (f"<tr><td>{r['idx']}</td><td>{r['truth']}</td>"
                     f"<td class={cls}>{r['pred']} {'✓' if r['correct'] else '✗'}</td><td>{r['p']:.3f}</td></tr>")
        body += "</table>"
    return f"<!doctype html><html><head><meta charset=utf-8>{head}</head><body>{body}</body></html>"

with open(f"{ART}/demo.html", "w") as f:
    f.write(_html(rows))
print("\nsaved: demo_predictions.json, demo.html")
